In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# load the pdf files
loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
raw_docs = loader.load()

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_29338/844840826.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rahultiwari/Documents/02_Freelancing/Hachion_batch/ai_engineering_19th_may/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# print(raw_docs[0].page_content)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100,
                                          separators=["\n\n", "\n", " ", ""])

chunks = splitter.split_documents(raw_docs)


In [3]:
print(chunks[0].page_content)

BAJAJ FINANCE LIMITED
 Helpdesk Agent Knowledge Base
 Prose Reference Edition — FY 2024–25
 Document Type: Internal Training & Reference Manual
 Coverage: Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy
 Intended Users: Helpdesk Agents · Branch Executives · Collections Team
 Classification: CONFIDENTIAL — For Internal Use Only
 Version: v1.0 Prose Edition — May 2025
This document is the prose-format reference edition of the Bajaj Finance Helpdesk Knowledge Base. All
policy information is presented in descriptive paragraph form to support agent training, onboarding, and
the BajajBot AI assistant knowledge base. For structured lookup tables, refer to the companion Policy
Reference Document v4.0.
Section 1 — Personal Loan: Eligibility, Rates & Charges
1.1 Who Can Apply for a Bajaj Finance Personal Loan
Bajaj Finance personal loans are designed for both salaried employees and self-employed


In [4]:
print(chunks[1].page_content)

Bajaj Finance personal loans are designed for both salaried employees and self-employed
professionals. However, all eligibility conditions must be satisfied simultaneously — meeting only some
of the criteria is not sufficient for approval.
For salaried applicants, the minimum age is 21 years and the maximum is 60 years at the time of
application. Self-employed professionals may apply between the ages of 25 and 65 years. The
minimum net take-home income required for salaried applicants is Rs 25,000 per month. Self-employed
professionals must demonstrate a minimum annual income of Rs 4.8 lakhs as declared in their Income
Tax Return.
A CIBIL score of 700 or above is required for standard approval. Applicants with a score between 650
and 699 may be considered under enhanced scrutiny, but approval is not guaranteed. Applicants with a
CIBIL score below 650 are automatically rejected by the system and cannot proceed with a personal


In [5]:
# print(chunks[2].page_content)
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [7]:
# pip install langchain-pinecone
# pip install pinecone

In [8]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

In [9]:
pc = Pinecone()

In [10]:
# create the index
# pc.list_indexes().names()
if "bajajbot-policy" not in pc.list_indexes().names():
    pc.create_index("bajajbot-policy", 
                    dimension=1536, 
                    metric="cosine",
                    spec=ServerlessSpec(cloud="aws",region="us-east-1"))

In [11]:
vectorstore = PineconeVectorStore.from_documents(chunks, embeddings,index_name="bajajbot-policy")

In [12]:
query = embeddings.embed_query("Bajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.")

# results = vectorstore.similarity_search(query)
index = pc.Index("bajajbot-policy")
stats = index.describe_index_stats()

In [13]:
stats

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 94}},
 'total_vector_count': 94,
 'vector_type': 'dense'}

In [14]:
result = index.fetch(ids=['06cb1951-3fb9-4014-87aa-ce7ab58f7fad'])

In [15]:
result

FetchResponse(namespace='', vectors={'06cb1951-3fb9-4014-87aa-ce7ab58f7fad': Vector(id='06cb1951-3fb9-4014-87aa-ce7ab58f7fad', values=[0.0144577026, 0.0515441895, 0.079284668, 0.0192871094, 0.0296630859, -0.000287294388, -0.002161026, -0.00761032104, -0.0164031982, 0.0621948242, 0.00617218, -0.0407104492, -0.0166931152, -0.0497741699, -0.00345802307, 0.0160675049, 0.0231170654, -0.00568389893, 0.00528717041, -0.00147342682, 0.0052986145, 0.0458984375, -0.0348205566, -0.00936889648, 0.00231742859, -0.0390930176, 0.00374984741, -0.0569458, 0.0169372559, -0.0169525146, 0.0386962891, 0.0073890686, 0.000394821167, 0.0333862305, -0.048614502, 0.029510498, 0.0069732666, 0.00532150269, 0.0291748047, -0.0282287598, -0.0386047363, -0.0299072266, -0.0252532959, -0.0176696777, -0.0201873779, 0.0390625, 0.0277557373, -0.00580978394, -0.0227508545, 0.0130157471, 0.00065279007, -0.0101089478, -0.000442743301, 0.00333786, -0.0267028809, 0.0213775635, -0.0362548828, 0.0143127441, -0.0463867188, -0.0392

In [16]:
query = embeddings.embed_query("Bajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.")


In [17]:
result = index.query(
    vector = query,
    top_k = 3,
    include_metadata=True
)

In [18]:
result.matches[0].metadata['text']

'Bajaj Finance personal loans are designed for both salaried employees and self-employed\nprofessionals. However, all eligibility conditions must be satisfied simultaneously — meeting only some\nof the criteria is not sufficient for approval.\nFor salaried applicants, the minimum age is 21 years and the maximum is 60 years at the time of\napplication. Self-employed professionals may apply between the ages of 25 and 65 years. The\nminimum net take-home income required for salaried applicants is Rs 25,000 per month. Self-employed\nprofessionals must demonstrate a minimum annual income of Rs 4.8 lakhs as declared in their Income\nTax Return.\nA CIBIL score of 700 or above is required for standard approval. Applicants with a score between 650\nand 699 may be considered under enhanced scrutiny, but approval is not guaranteed. Applicants with a\nCIBIL score below 650 are automatically rejected by the system and cannot proceed with a personal'

In [19]:
result.matches[1].metadata['text']

'Bajaj Finance personal loans are designed for both salaried employees and self-employed\nprofessionals. However, all eligibility conditions must be satisfied simultaneously — meeting only some\nof the criteria is not sufficient for approval.\nFor salaried applicants, the minimum age is 21 years and the maximum is 60 years at the time of\napplication. Self-employed professionals may apply between the ages of 25 and 65 years. The\nminimum net take-home income required for salaried applicants is Rs 25,000 per month. Self-employed\nprofessionals must demonstrate a minimum annual income of Rs 4.8 lakhs as declared in their Income\nTax Return.\nA CIBIL score of 700 or above is required for standard approval. Applicants with a score between 650\nand 699 may be considered under enhanced scrutiny, but approval is not guaranteed. Applicants with a\nCIBIL score below 650 are automatically rejected by the system and cannot proceed with a personal'

In [20]:
result.matches[2].metadata['text']

'LLPs, and Public Limited Companies. The minimum business vintage is three years in profit for most\nsectors, extending to five years for manufacturing businesses.\n6.2 Sector-Specific Rules\nBajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.\nThis reflects the varying risk profiles and cash flow patterns across industries.\nTrading and retail businesses require a minimum vintage of three years and may borrow up to Rs 50\nlakhs without collateral. GST returns are mandatory for this sector. For loan amounts above Rs 25 lakhs\nin this segment, a stock audit is required before disbursement.\nService businesses including IT firms and consulting companies also require three years of vintage and\nare eligible for up to Rs 50 lakhs unsecured. For loans above Rs 20 lakhs in this segment, client\ncontracts must be provided as supplementary income proof.'

In [21]:
context = [
    result.matches[0].metadata['text'],
    result.matches[1].metadata['text'],
    result.matches[2].metadata['text']
]

# convert into string
context = "\n".join(context)


In [22]:
from langchain_openai import ChatOpenAI

In [29]:
template = """
You are a financial expert helping users 
understand their eligibility for loans. 
Below is the context information you can use to answer the user's question:

{context}

User's question: {questions}
"""
prompt = template.format(context=context, questions="Bajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.")

llm = ChatOpenAI(model="gpt-4o-mini")

response = llm.invoke(prompt).content

In [31]:
print(response)

Yes, Bajaj Finance does apply different eligibility criteria depending on the sector in which the applicant operates. This is because various sectors present different risk profiles and cash flow patterns, which can affect the eligibility for loans. 

Here are some key points regarding sector-specific rules:

1. **Trading and Retail Businesses**:
   - Applicants must have a minimum business vintage of three years.
   - They may borrow up to Rs 50 lakhs without collateral.
   - GST returns are mandatory for this sector.
   - For loan amounts exceeding Rs 25 lakhs, a stock audit is required before disbursement.

2. **Service Businesses (e.g., IT firms, consulting companies)**:
   - A minimum business vintage of three years is also required.
   - They are eligible for unsecured loans of up to Rs 50 lakhs.
   - For loans above Rs 20 lakhs, client contracts must be provided as supplementary proof of income.

These specific eligibility rules help Bajaj Finance assess the financial stability 

In [26]:
print(prompt)


You are a financial expert helping users 
understand their eligibility for loans. 
Below is the context information you can use to answer the user's question:

Bajaj Finance personal loans are designed for both salaried employees and self-employed
professionals. However, all eligibility conditions must be satisfied simultaneously — meeting only some
of the criteria is not sufficient for approval.
For salaried applicants, the minimum age is 21 years and the maximum is 60 years at the time of
application. Self-employed professionals may apply between the ages of 25 and 65 years. The
minimum net take-home income required for salaried applicants is Rs 25,000 per month. Self-employed
professionals must demonstrate a minimum annual income of Rs 4.8 lakhs as declared in their Income
Tax Return.
A CIBIL score of 700 or above is required for standard approval. Applicants with a score between 650
and 699 may be considered under enhanced scrutiny, but approval is not guaranteed. Applicants with 